# Stage 4 — Scoring QC

Inspect the score distribution and top-N alert cards after running the full pipeline.

| Step | What | What to check |
|------|------|---------------|
| 0 | Config | Dates |
| 1 | Session + full pipeline | Alert cards list |
| 2 | Score distribution | Range, percentiles |
| 3 | Top-N cards | Highest priority alerts |
| 4 | Trigger breakdown | Scores by trigger type |
| 5 | Write to Snowflake | **Run manually when satisfied** |

In [ ]:
ANALYSIS_DATE_STR = '2025-06-14'

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None   # or 'eastofengland'

QC_STORE_ID       = None
TOP_N             = 10

import datetime
analysis_date   = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))
RUN_DATE        = str(datetime.date.today())

print(f'Analysis date: {THIS_WEEK_END}  |  Run date: {RUN_DATE}')
print(f'Client: {CLIENT or "base defaults"}')

In [ ]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

# ── Suppress noisy Snowflake connector output ─────────────────────────────────
logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.alerts.config.loader import load_config
from store_insight.alerts.pipeline.session import create_session
from store_insight.alerts.pipeline.pipeline import run_pipeline
from snowflake.snowpark import functions as F

cfg     = load_config(CLIENT)
print(f'Config loaded: client={cfg["client_name"]}')

session = create_session(cfg)   # prints 'Snowflake session created.'

# ── Resolve table names from config ──────────────────────────────────────────
OSA_TABLE       = cfg['tables']['osa']
HIERARCHY_TABLE = cfg['tables']['hierarchy']

print(f'OSA table       : {OSA_TABLE}')
print(f'Hierarchy table : {HIERARCHY_TABLE}')

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('ANALYSIS_DATE') >= LOOKBACK_START)
    .filter(F.col('ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'), F.col('ITEM_ID').alias('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'), F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'), F.col('ANALYSIS_DATE'), F.col('DEPARTMENT_ID'),
    )
)
item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('MASTER_ITEM_ID').alias('ITEM_CODE'), F.col('ITEM_ID'),
        F.col('DEPARTMENT_NAME'), F.col('SUBCLASS_NAME'), F.col('PRODUCT_PROFILE_GROUP_NAME'),
    ).distinct()
)
osa = (
    osa_raw.join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('DEPARTMENT_NAME').is_not_null())
    .cache_result()
)

cards = run_pipeline(
    osa,
    last_week_start=LAST_WEEK_START, last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    store_id=QC_STORE_ID,
    session=session,
    cfg=cfg,
)
print(f'{len(cards)} alert cards produced')

## Step 2 — Score Distribution

In [ ]:
import statistics

scores = [c['sort_value'] for c in cards]
print(f'Count  : {len(scores)}')
print(f'Min    : {min(scores):,.2f}')
print(f'Max    : {max(scores):,.2f}')
print(f'Median : {statistics.median(scores):,.2f}')
print(f'Mean   : {statistics.mean(scores):,.2f}')

# Decile buckets
sorted_scores = sorted(scores)
n = len(sorted_scores)
print('\nDeciles:')
for pct in range(10, 110, 10):
    idx = min(int(n * pct / 100), n - 1)
    print(f'  P{pct:>3}  £{sorted_scores[idx]:>10,.2f}')

## Step 3 — Top-N Alert Cards

In [ ]:
print(f'Top {TOP_N} alert cards by score:')
print(f"  {'store':>5}  {'dept':<30}  {'triggers':<25}  {'score':>10}")
print(f"  {'-'*5}  {'-'*30}  {'-'*25}  {'-'*10}")
for c in cards[:TOP_N]:
    triggers = '+'.join(c['trigger_types'])
    print(f"  {c['store_id']:>5}  {c['section']:<30}  {triggers:<25}  {c['sort_value']:>10,.2f}")

# Print full narrative for rank 1
top = cards[0]
print(f"\n{'='*60}")
print(f"RANK 1 NARRATIVE — store={top['store_id']}  dept={top['section']}")
print(f"{'='*60}")
print(f"ISSUE:\n  {top['issue']}")
print(f"\nEXPLANATION:\n{top['explanation']}")
print(f"\nCOST IF NOT FIXED:\n  {top['cost_if_not_fixed']}")
print(f"\nRECOMMENDED ACTION:\n  {top['recommended_action']}")

## Step 4 — Score by Trigger Combination

In [ ]:
from collections import defaultdict

by_trigger = defaultdict(list)
for c in cards:
    key = '+'.join(sorted(c['trigger_types']))
    by_trigger[key].append(c['sort_value'])

print(f"{'Triggers':<30}  {'Count':>5}  {'Avg Score':>10}  {'Max Score':>10}")
print(f"{'-'*30}  {'-'*5}  {'-'*10}  {'-'*10}")
for trigger, sc in sorted(by_trigger.items(), key=lambda x: statistics.mean(x[1]), reverse=True):
    print(f"{trigger:<30}  {len(sc):>5}  {statistics.mean(sc):>10,.2f}  {max(sc):>10,.2f}")

## Step 5 — Write to Snowflake

> **Only run this cell when you are happy with the output above.**
> Set `OVERWRITE = True` to replace existing data for this run date.

In [ ]:
OVERWRITE = False   # set True to overwrite

from store_insight.alerts.pipeline.pipeline import write_alerts
write_alerts(session, cards, run_date=RUN_DATE, overwrite=OVERWRITE, cfg=cfg)